# 03 — Predict & Visualise Channel Unmixing

Runs MicroSplit prediction on the demo dataset and visualises the unmixing results.

**Run notebooks 00–02 first.**

In [ ]:
# imports
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
import torch

sys.path.insert(0, '../../../src')
from microsplit_reproducibility.workflows.cellpainting import (
    find_checkpoint,
    load_model_and_stats,
    predict_fov,
    compute_metrics,
    save_fov_predictions,
)

In [ ]:


DEMO_DIR    = Path('./cpg0000_demo')
DATASET_DIR = DEMO_DIR / 'dataset'
PRED_DIR    = DEMO_DIR / 'predictions'
CHANNELS    = ['DNA', 'RNA', 'ER', 'AGP', 'Mito']

if not DATASET_DIR.exists():
    raise FileNotFoundError(f'{DATASET_DIR} not found — run earlier notebooks first.')

PRED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
checkpoint = find_checkpoint(str(DATASET_DIR))
model, stats = load_model_and_stats(
    training_dir=str(DATASET_DIR),
    checkpoint_path=checkpoint,
    channel_names=CHANNELS,
)
print(f'Checkpoint: {Path(checkpoint).name}  |  device: {next(model.parameters()).device}')

In [ ]:
image_id = 0
channel_images = {
    ch: tifffile.imread(str(DATASET_DIR / ch / f'{image_id:06d}.tiff'))
    for ch in CHANNELS
}

mmse_pred, posterior_samples = predict_fov(
    model=model,
    channel_images=channel_images,
    stats=stats,
    channel_names=CHANNELS,
    image_size=64,
    grid_size=8,
    multiscale_lowres_count=3,
    mmse_count=10,
    num_posterior_samples=1,
    posterior_seeds=(42,),
    batch_size=16,
)

In [ ]:
combined = tifffile.imread(str(DATASET_DIR / 'combined' / f'{image_id:06d}.tiff'))

n = len(CHANNELS)
fig, axes = plt.subplots(2, n + 1, figsize=(4 * (n + 1), 8))

axes[0, 0].imshow(combined, cmap='viridis', vmin=0, vmax=np.percentile(combined, 99.5))
axes[0, 0].set_title('Combined input')
axes[0, 0].axis('off')
axes[1, 0].axis('off')
axes[1, 0].text(0.5, 0.5, 'MicroSplit\nMMSE', ha='center', va='center', fontsize=12)

for c_idx, ch in enumerate(CHANNELS):
    gt   = channel_images[ch].astype(float)
    vmax = np.percentile(gt, 99.5) if gt.max() > 0 else 1
    axes[0, c_idx + 1].imshow(gt, cmap='gray', vmin=0, vmax=vmax)
    axes[0, c_idx + 1].set_title(f'GT {ch}')
    axes[0, c_idx + 1].axis('off')
    axes[1, c_idx + 1].imshow(mmse_pred[:, :, c_idx], cmap='gray', vmin=0, vmax=vmax)
    axes[1, c_idx + 1].set_title(f'Pred {ch}')
    axes[1, c_idx + 1].axis('off')

plt.suptitle(f'MicroSplit — combined → channels  (image_id={image_id}, 5 epochs)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
metrics = compute_metrics(channel_images, mmse_pred, CHANNELS)
df = pd.DataFrame([
    {'channel': ch, 'PSNR': metrics[f'psnr_{ch}'], 'SSIM': metrics[f'ssim_{ch}']}
    for ch in CHANNELS
])
print(df.to_string(index=False))
print(f"\nMean PSNR: {df['PSNR'].mean():.2f} dB   Mean SSIM: {df['SSIM'].mean():.4f}")

save_fov_predictions(
    output_dir=str(PRED_DIR),
    well='demo',
    site=1,
    mmse_prediction=mmse_pred,
    posterior_samples=posterior_samples,
    channel_names=CHANNELS,
    save_uint16=True,
)
print(f'Predictions saved → {PRED_DIR}')